# Simple BERTopic Pipeline (Hippocorpus)

1. Run BERTopic on the full Hippocorpus.
2. Reduce outliers and export topics for inspection.
3. Apply your inspected regex rules to label the full Hippocorpus.
4. Append labels to reduced datasets by statement index.

In [ ]:
from pathlib import Path
import re
import pandas as pd
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
DATA_DIR = Path('../data')
HIPPOCORPUS_PATH = DATA_DIR / 'hippocorpus.csv'

# Topic fitting mode: 'hdbscan' (default) or 'kmeans'
TOPIC_MODE = 'hdbscan'
KMEANS_N_CLUSTERS = 30

OUTPUT_DOCS_PATH = DATA_DIR / ('output_k_means.csv' if TOPIC_MODE == 'kmeans' else 'output.csv')
OUTPUT_TOPICS_PATH = DATA_DIR / ('topic_info_k_means.csv' if TOPIC_MODE == 'kmeans' else 'topic_info.csv')
LABELED_HIPPOCORPUS_PATH = DATA_DIR / 'hippocorpus_labeled.csv'

# Add or remove targets here whenever you want to append labels to new files.
APPEND_TARGETS = [
    {'input': 'final.csv', 'index_col': 'os_id', 'output': 'final_labeled.csv'},
    {'input': 'final_deltamax.csv', 'index_col': 'os_id', 'output': 'final_deltamax_labeled.csv'},
]

hip = pd.read_csv(HIPPOCORPUS_PATH, sep=',')
docs = hip['text'].astype(str).tolist()
print(f'Hippocorpus rows: {len(hip):,}')

## Step 1 and 2: Fit topics
- `TOPIC_MODE = 'hdbscan'`: BERTopic + outlier reduction.
- `TOPIC_MODE = 'kmeans'`: BERTopic with fixed KMeans clusters (no outlier reduction).

In [ ]:
if TOPIC_MODE == 'kmeans':
    cluster_model = KMeans(n_clusters=KMEANS_N_CLUSTERS, random_state=42)
    vectorizer_model = CountVectorizer(stop_words='english')
    representation_model = KeyBERTInspired()

    topic_model = BERTopic(
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,
    )
    topics, probs = topic_model.fit_transform(docs)

    # Old kmeans pipeline: no outlier reduction step
    hip['topic'] = topics
else:
    representation_model = KeyBERTInspired()
    topic_model = BERTopic(representation_model=representation_model)
    topics, probs = topic_model.fit_transform(docs)

    # Old hdbscan pipeline: reduce outliers then update topic representations
    reduced_topics = topic_model.reduce_outliers(docs, topics)
    hip['topic'] = reduced_topics
    topic_model.update_topics(docs, topics=reduced_topics)

hip.to_csv(OUTPUT_DOCS_PATH, index=False)
topic_info = topic_model.get_topic_info()
topic_info.to_csv(OUTPUT_TOPICS_PATH, index=False)

print(f'Saved: {OUTPUT_DOCS_PATH}')
print(f'Saved: {OUTPUT_TOPICS_PATH}')
topic_info.head(10)

## Step 3: Create regex rules from inspected topics and label full Hippocorpus
Edit only `RULES` below. First match wins.

In [ ]:
RULES = [
    (1, 'Funeral/Death', re.compile(
        r'\b(funeral|wake|burial|cremat(?:e|ion|ed)|cemeter(?:y|ies)|died|death|passed away|grief|mourning|eulog(?:y|ies)|memorial)\b',
        re.I,
    )),
    (2, 'Hospital/Injury', re.compile(
        r'\b(hospital|doctor|dr\.?|nurse|clinic|er|emergency(?: room)?|ambulance|surgery|operation|radiation|chemo(?:therapy)?|cancer|tumou?r|treatment|therapy|medication|prescription|diagnos(?:is|ed)|pain|injur(?:y|ed)|hurt|wound|bleed(?:ing)?|stitch(?:es)?|broken|fracture|sprain|concussion|infection|fever)\b',
        re.I,
    )),
    (3, 'Money/Gambling', re.compile(
        r'\b(casino|gambl(?:e|ing)|slot(?:s)?|bet(?:s|ting)?|jackpot|poker|blackjack|roulette|lottery|winnings?|won (?:big|money)|debt|loan|overdraft|credit card|bank account|bills?|payment(?:s)?)\b',
        re.I,
    )),
    (4, 'NaturalDisaster/Weather', re.compile(
        r'\b(hurricane|tornado|earthquake|aftershock|wildfire|evacuat(?:e|ed|ion)|flood(?:ed|ing)?|storm surge|blizzard|ice storm|hailstorm|water damage|house flooded)\b',
        re.I,
    )),
    (5, 'Children', re.compile(
        r'\b(baby|newborn|toddler|child|children|kid|kids|pregnan(?:t|cy)|expecting|labor|delivery|gave birth|born|birth|ultrasound|due date|midwife|diaper(?:s)?|daycare|crib|stroller)\b',
        re.I,
    )),
    (6, 'Pets', re.compile(
        r'\b(dog|puppy|cat|kitten|pet|pets|vet|veterinarian|animal hospital|leash|litter|adopt(?:ed|ion)?|groom(?:ed|er|ing)?|walk(?:ed|ing)? (?:the )?dog)\b',
        re.I,
    )),
    (7, 'Vacation/Trip', re.compile(
        r'\b(vacation|holiday|trip|road trip|travel(?:ed|ing)?|flight|airport|hotel|resort|airbnb|beach|mountain(?:s)?|hike(?:d|ing)?|trail|camp(?:ing)?|cabin|lake|river|kayak(?:ing)?|raft(?:ing)?|fishing|bait|national park|ski(?:ing)?|snowboard(?:ing)?|surf(?:ing)?|paris|london|europe|new york)\b',
        re.I,
    )),
    (8, 'Work', re.compile(
        r'\b(job|work|company|boss|manager|coworker(?:s)?|colleague(?:s)?|office|shift|meeting(?:s)?|deadline|project|client|promotion|promoted|raise|salary|position|appl(?:y|ied|ication)|interview(?:s)?|resume|cv|hired|fired|laid off|quit|resign(?:ed|ation)?)\b',
        re.I,
    )),
    (9, 'Birthday', re.compile(
        r'\b(birthday|bday|surprise party|party|cake|candles|balloons|present(?:s)?|gift(?:s)?|blew out (?:the )?candles)\b',
        re.I,
    )),
    (10, 'Sports', re.compile(
        r'\b(playoffs|tournament|match|season|team|coach|practice|race|marathon|triathlon|training|workout|gym|baseball|basketball|soccer|football|tennis|hockey|swim(?:ming)?|cycling|running|lift(?:ing)?|weights?)\b',
        re.I,
    )),
    (11, 'Concert/Music', re.compile(
        r'\b(concert|gig|live music|show|band|music festival|festival|setlist|encore|venue|tickets?)\b',
        re.I,
    )),
    (12, 'School', re.compile(
        r'\b(university|college|school|campus|class(?:es)?|lecture(?:s)?|teacher|professor|homework|assignment(?:s)?|exam(?:s)?|test(?:s)?|final(?:s)?|graduat(?:e|ed|ion)|degree|diploma|ceremony)\b',
        re.I,
    )),
    (13, 'Home/Moving', re.compile(
        r'\b(house|home|apartment|flat|condo|roommate(?:s)?|mortgage|lease|rent|landlord|tenant|moved in|move(?:d|s|ing)?|moving|packing|boxes|unpack(?:ed|ing)?|new place|new house|new apartment)\b',
        re.I,
    )),
    (14, 'Relationships', re.compile(
        r'\b(wife|husband|partner|fianc(?:e|ée)|boyfriend|girlfriend|dating|date night|single|break(?:\s?up)?|broke up|marry|married|wedding|engag(?:e|ed|ement)|proposal|propos(?:e|al)|ring|in love|love)\b',
        re.I,
    )),
    (15, 'Family', re.compile(
        r'\b(mom|mother|mum|dad|father|parents?|stepmom|stepmother|stepdad|stepfather|sister|brother|siblings?|daughter|son|grandma|grandpa|grandmother|grandfather|aunt|uncle|cousin(?:s)?|niece|nephew|in-?laws?|family reunion|custody)\b',
        re.I,
    )),
    (16, 'Writing', re.compile(
        r'\b(diary|journal|journaling|dear diary|book(?:s)?|novel|chapter|story|writ(?:e|ing|ten)|author|manuscript|publish(?:ed|ing)?|poem(?:s)?|poetry|library)\b',
        re.I,
    )),
]

def label_text(text):
    if pd.isna(text):
        return 0, 'Other'
    for topic_id, topic_label, pattern in RULES:
        if pattern.search(str(text)):
            return topic_id, topic_label
    return 0, 'Other'

In [ ]:
hip = pd.read_csv(HIPPOCORPUS_PATH)
hip[['topic_id', 'topic_label']] = hip['text'].apply(lambda x: pd.Series(label_text(x)))
hip.to_csv(LABELED_HIPPOCORPUS_PATH, index=False)

print(f'Saved: {LABELED_HIPPOCORPUS_PATH}')
hip['topic_label'].value_counts(dropna=False).head(20)

## Step 4: Append labels to reduced datasets by statement index

In [ ]:
hip_labeled = pd.read_csv(LABELED_HIPPOCORPUS_PATH)

# Statement index alignment in the labeled Hippocorpus
hip_labeled['index'] = hip_labeled['index'].astype(str)
labels = hip_labeled[['index', 'topic_id', 'topic_label']].drop_duplicates(subset='index')

for target in APPEND_TARGETS:
    input_path = DATA_DIR / target['input']
    output_path = DATA_DIR / target['output']
    index_col = target['index_col']

    reduced = pd.read_csv(input_path)
    reduced[index_col] = reduced[index_col].astype(str)

    merged = reduced.merge(labels, left_on=index_col, right_on='index', how='left').drop(columns=['index'])
    missing = merged['topic_label'].isna().sum()

    merged.to_csv(output_path, index=False)
    print(f'Saved: {output_path} | Missing labels: {missing}')

Outputs:
- data/output.csv
- data/topic_info.csv
- data/hippocorpus_labeled.csv
- one labeled output per APPEND_TARGETS entry